# A Catholic Introduction to Artificial Intelligence
## Final Project — Module 2: Deeper Patterns & Feature Engineering
### Classes 3 & 4

---

> *"Ethical discernment cannot be limited to asking whether we are using a system for good or bad purposes; it must also examine how that system is designed and what vision of the human person and society is embedded in the data and models that guide it."*
> — Pope Leo XIV, *Magnifica Humanitas*, no. 104

---

## Module Overview

In Module 1, you explored the dataset and developed an initial sense of its structure and patterns. In this module, you will go deeper — investigating which variables are most strongly associated with on-time submission, creating new features that capture ideas the raw data does not directly measure, and beginning to ask a harder question: **which of these features should an ethical AI system be permitted to use?**

### What You Will Do in This Module

1. Investigate patterns across student groups and assignment characteristics in more detail
2. Create new **engineered features** — variables calculated from the existing data that may be more informative for prediction
3. Evaluate which features are ethically appropriate to include in a predictive model
4. Produce a documented **feature analysis report** summarizing your findings

---

## Setup: Load Data and Module 1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('synthetic_homework_dataset.csv',
                 parse_dates=['date_assigned', 'date_submitted'])

print(f'Dataset loaded: {len(df)} rows, {df.columns.tolist()}')

---

## Part 1: Who Is Being Predicted?

Before we analyze features, we should understand the **distribution of outcomes across students**. Remember that our model will eventually make a prediction about each student-assignment pair. If certain students are almost always late or almost always on time, the model's job will be easier for them — but the model may also disproportionately label some students as "at risk" regardless of the specific assignment.

This is a pattern that appears in many real-world AI systems: past behavior is used to predict future behavior, in ways that can become self-reinforcing.

In [ ]:
# Summarize each student's overall completion rate
student_summary = df.groupby('student_id').agg(
    total_assignments=('completed_on_time', 'count'),
    on_time_rate=('completed_on_time', 'mean'),
    avg_grade=('prior_avg_grade', 'mean'),
    avg_difficulty_faced=('difficulty', 'mean'),
    avg_time_spent=('time_spent_minutes', 'mean')
).round(2).reset_index()

# Categorize students by on-time rate
def tier(rate):
    if rate >= 0.90: return 'High (90–100%)'
    elif rate >= 0.75: return 'Medium (75–89%)'
    else: return 'Low (<75%)'

student_summary['tier'] = student_summary['on_time_rate'].apply(tier)

tier_counts = student_summary['tier'].value_counts()
print('Student tiers:')
print(tier_counts)
print()
print('Students always on time:', (student_summary['on_time_rate'] == 1.0).sum())
print('Students never on time: ', (student_summary['on_time_rate'] == 0.0).sum())
print('Students with mixed record:', ((student_summary['on_time_rate'] > 0) & (student_summary['on_time_rate'] < 1)).sum())

In [ ]:
# Visualize the spread of on-time rates across students
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: distribution
axes[0].hist(student_summary['on_time_rate'], bins=12,
             color='#5C8BC7', edgecolor='white')
axes[0].set_xlabel('Student On-Time Completion Rate')
axes[0].set_ylabel('Number of Students')
axes[0].set_title('Distribution of On-Time Rates\nAcross All Students', fontweight='bold')
axes[0].axvline(student_summary['on_time_rate'].mean(), color='red',
                linestyle='--', label=f'Mean: {student_summary["on_time_rate"].mean():.2f}')
axes[0].legend()

# Right: tier breakdown
tier_order = ['High (90–100%)', 'Medium (75–89%)', 'Low (<75%)']
tier_vals = [tier_counts.get(t, 0) for t in tier_order]
colors = ['#4CAF50', '#FFC107', '#E57373']
axes[1].barh(tier_order, tier_vals, color=colors)
axes[1].set_xlabel('Number of Students')
axes[1].set_title('Students by On-Time Rate Tier', fontweight='bold')
for i, v in enumerate(tier_vals):
    axes[1].text(v + 0.2, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### The Feedback Loop Problem

Notice that `prior_completion_rate` — a student's historical on-time rate — is one of the features in the dataset. If a student has been late in the past, this feature will flag them as higher risk in the future, even if their circumstances have changed.

This is a real and documented problem in AI systems. It is sometimes called a **feedback loop**: the AI's predictions, if acted upon, can reinforce the very patterns the model identified, making it harder for individuals to escape a label once assigned.

> 🤔 **Think about it:** Imagine a student who had a difficult semester last year (family illness, personal crisis) and was late on several assignments as a result. They are now doing much better. Should their historical completion rate still count against them in a prediction model? What Catholic principle does this question invoke?

---

## Part 2: Deeper Pattern Analysis

Let's investigate more carefully which factors are associated with late submission.

In [ ]:
# --- Does prior grade predict on-time completion? ---
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

if 'df' not in globals():
    data_path = Path('synthetic_homework_dataset.csv')
    if not data_path.exists():
        data_path = Path('Final Project') / 'synthetic_homework_dataset.csv'
    df = pd.read_csv(data_path, parse_dates=['date_assigned', 'date_submitted'])

# Create grade buckets for easier analysis
df['grade_bucket'] = pd.cut(df['prior_avg_grade'],
                             bins=[0, 70, 80, 90, 101],
                             labels=['<70', '70–79', '80–89', '90–100'])

grade_completion = df.groupby('grade_bucket', observed=True)['completed_on_time'].agg(['mean','count'])
grade_completion.columns = ['on_time_rate', 'count']

fig, ax = plt.subplots()
bars = ax.bar(grade_completion.index, grade_completion['on_time_rate'] * 100,
              color=['#E57373','#FFB74D','#81C784','#4CAF50'])
ax.set_xlabel('Prior Average Grade Bucket')
ax.set_ylabel('On-Time Completion Rate (%)')
ax.set_title('On-Time Completion Rate by Prior Grade', fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
for bar, (_, row) in zip(bars, grade_completion.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, row['on_time_rate']*100 + 1.5,
            f"{row['on_time_rate']*100:.1f}%\n(n={int(row['count'])})",
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

This chart suggests that turning work in on time and earning high grades are not exactly the same thing. Some students consistently submit work on time but still struggle with the material. Other students earn high grades but occasionally submit work late.

Grades are not a very strong predictor of whether work gets turned in on time.

Time management and academic performance are related but distinct skills. Some students struggle with content; others struggle with deadlines; some struggle with both; and some excel at both.

In [ ]:
# --- Does the amount of advance notice affect on-time submission? ---
due_completion = df.groupby('days_until_due')['completed_on_time'].agg(['mean','count'])
due_completion.columns = ['on_time_rate', 'count']

fig, ax = plt.subplots()
ax.bar(due_completion.index, due_completion['on_time_rate'] * 100,
       color='#7986CB', width=0.6)
ax.set_xlabel('Days Until Due (advance notice given)')
ax.set_ylabel('On-Time Completion Rate (%)')
ax.set_title('Does More Advance Notice Help Students Submit on Time?',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
ax.set_xticks(due_completion.index)
for i, (idx, row) in enumerate(due_completion.iterrows()):
    ax.text(idx, row['on_time_rate']*100 + 1.5,
            f"{row['on_time_rate']*100:.0f}%", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(due_completion.round(3))

In [ ]:
# --- Prior completion rate buckets ---
df['prior_bucket'] = pd.cut(df['prior_completion_rate'],
                             bins=[0, 0.60, 0.75, 0.90, 1.01],
                             labels=['<60%', '60–75%', '75–90%', '>90%'])

prior_completion = df.groupby('prior_bucket', observed=True)['completed_on_time'].agg(['mean','count'])
prior_completion.columns = ['on_time_rate', 'count']

fig, ax = plt.subplots()
bars = ax.bar(prior_completion.index, prior_completion['on_time_rate'] * 100,
              color=['#E57373','#FFB74D','#81C784','#4CAF50'])
ax.set_xlabel('Prior Completion Rate (historical)')
ax.set_ylabel('Current On-Time Rate (%)')
ax.set_title('How Strongly Does Past Behavior Predict Present Behavior?',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 115)
for bar, (_, row) in zip(bars, prior_completion.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, row['on_time_rate']*100 + 1.5,
            f"{row['on_time_rate']*100:.1f}%\n(n={int(row['count'])})",
            ha='center', fontsize=9)
plt.tight_layout()
plt.show()

This chart is about habits. We grouped students based on their past record of turning work in on time. Then we looked at whether they turned in the current assignment on time.

Students with the strongest history of meeting deadlines continued to meet deadlines about 95% of the time. Students with the weakest history met the deadline only about 40% of the time.

In other words, your past habits are one of the best predictors of your future habits.

In [ ]:
# --- Assignment type × difficulty heatmap ---
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if 'df' not in globals():
    data_path = Path('synthetic_homework_dataset.csv')
    if not data_path.exists():
        data_path = Path('Final Project') / 'synthetic_homework_dataset.csv'
    df = pd.read_csv(data_path, parse_dates=['date_assigned', 'date_submitted'])

pivot = df.groupby(['assignment_type', 'difficulty'])['completed_on_time'].mean().unstack()

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn', vmin=0.6, vmax=1.0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'On-Time Rate'})
ax.set_title('On-Time Completion Rate by Assignment Type and Difficulty',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Difficulty Level (1=Easy, 5=Hard)')
ax.set_ylabel('Assignment Type')
plt.tight_layout()
plt.show()

### What the Heatmap Shows

The heatmap reveals that the relationship between difficulty and on-time completion is **not simple or uniform**. Different assignment types respond differently to difficulty changes — essays at difficulty 2 have a 100% on-time rate, while readings at difficulty 1 are among the lower performers.

This is an important finding: a simple model that treats difficulty as straightforwardly predictive of lateness would miss this nuance. Real students respond to different types of work in different ways.

> 🤔 **Think about it:** Can you think of reasons why *easy* assignments might sometimes have *lower* completion rates? What does this tell us about what the data is actually measuring?

This chart suggests that students don't necessarily turn work in late because it's hard. Some difficult assignments have very high on-time rates, and some easier assignments have lower on-time rates.

We looked at grades, assignment difficulty, and assignment type. None of those explained on-time submissions very well. The strongest predictor was simple: students who had built a habit of turning work in on time were much more likely to continue doing so.

---

## Part 3: Feature Engineering

**Feature engineering** is the process of creating new variables from existing ones that may capture something more meaningful for prediction. It is one of the most important — and most creative — steps in building an AI system.

Raw data rarely gives you exactly what you need. Sometimes the most informative signal isn't a single column, but a relationship between two columns. For example:

- `time_spent_minutes` alone might not be very informative — a long essay takes more time than a short quiz regardless of effort
- But `time_spent_minutes / num_questions` (time per question) captures something more meaningful: **how deeply** the student engaged with each item

Let's create several engineered features.

In [ ]:
# Feature 1: Days actually taken to submit
# How many days elapsed between assignment date and submission date?
df['days_taken'] = (df['date_submitted'] - df['date_assigned']).dt.days

# Feature 2: Time per question
# How many minutes did the student spend per question/item?
df['time_per_question'] = df['time_spent_minutes'] / df['num_questions']

# Feature 3: Effort ratio
# How does time spent on THIS assignment compare to the student's historical average?
# > 1.0 means they spent more time than usual; < 1.0 means less
df['effort_ratio'] = df['time_spent_minutes'] / df['prior_avg_homework_time']

# Feature 4: Time pressure
# How difficult was this assignment relative to how much time was given?
df['time_pressure'] = df['difficulty'] / df['days_until_due']

# Feature 5: Deadline usage
# What fraction of the available time did the student actually use?
df['deadline_usage'] = df['days_taken'] / df['days_until_due']

print('New features created:')
new_features = ['days_taken', 'time_per_question', 'effort_ratio', 'time_pressure', 'deadline_usage']
print(df[['student_id', 'assignment_id'] + new_features + ['completed_on_time']].head(8).to_string())

In [ ]:
# How do the new features correlate with on-time completion?
all_features = [
    'num_questions', 'difficulty', 'days_until_due',
    'time_spent_minutes', 'num_work_sessions',
    'prior_completion_rate', 'prior_avg_grade', 'prior_avg_homework_time',
    'days_taken', 'time_per_question', 'effort_ratio',
    'time_pressure', 'deadline_usage'
]

correlations = df[all_features + ['completed_on_time']].corr()['completed_on_time'].drop('completed_on_time')
correlations = correlations.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#4CAF50' if v > 0 else '#E57373' for v in correlations.values]
bars = ax.barh(correlations.index, correlations.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlation with completed_on_time')
ax.set_title('Feature Correlations with On-Time Completion\n(including engineered features)',
             fontsize=13, fontweight='bold')

# Mark engineered features with an asterisk
engineered = {'days_taken', 'time_per_question', 'effort_ratio', 'time_pressure', 'deadline_usage'}
labels = [f'{name} *' if name in engineered else name for name in correlations.index]
ax.set_yticks(range(len(correlations)))
ax.set_yticklabels(labels)

for bar, val in zip(bars, correlations.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=8)

ax.text(0.98, 0.02, '* = engineered feature', transform=ax.transAxes,
        ha='right', fontsize=9, style='italic', color='gray')
plt.tight_layout()
plt.show()

We looked at many factors to see what predicts turning work in on time. Surprisingly, grades didn't matter much. Assignment difficulty didn't matter much. The strongest predictor was a student's past completion history. Students who had a habit of turning work in on time were the most likely to continue doing so.

In [ ]:
# Deep dive: deadline_usage by on-time status
# (capped at 2.0 to exclude extreme outliers for visualization)
plot_df = df[df['deadline_usage'] <= 2.0].copy()

fig, ax = plt.subplots()
for label, color, name in [(1, '#4CAF50', 'On time'), (0, '#E57373', 'Late')]:
    subset = plot_df[plot_df['completed_on_time'] == label]['deadline_usage']
    ax.hist(subset, bins=20, alpha=0.65, color=color, label=f'{name} (mean: {subset.mean():.2f})')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Due date = 1.0')
ax.set_xlabel('Deadline Usage (days taken / days available)')
ax.set_ylabel('Number of Submissions')
ax.set_title('Deadline Usage: On-Time vs. Late Submissions', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

The vertical line is the deadline. Everything to the left was submitted on time. Everything to the right was submitted late. Students who turned work in on time typically used about half of the available time. Students who turned work in late usually went well past the deadline.

The more interesting question is not whether students crossed the deadline. It's which students tend to cross the deadline. That's where we found that past completion habits were the strongest predictor.

In [ ]:
# Effort ratio: does spending more than usual help?
fig, ax = plt.subplots()
plot_df2 = df[df['effort_ratio'] <= 4.0].copy()  # cap outliers
for label, color, name in [(1, '#4CAF50', 'On time'), (0, '#E57373', 'Late')]:
    subset = plot_df2[plot_df2['completed_on_time'] == label]['effort_ratio']
    ax.hist(subset, bins=20, alpha=0.65, color=color, label=f'{name} (mean: {subset.mean():.2f})')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='Average effort = 1.0')
ax.set_xlabel('Effort Ratio (time spent ÷ historical average time)')
ax.set_ylabel('Number of Submissions')
ax.set_title('Effort Relative to Baseline: On-Time vs. Late', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

Students who turned work in late spent about the same amount of time working as students who turned it in on time.

So this isn't really a story about intelligence or effort. Many late assignments came from students who were working just as hard.

The difference seems to be when they started, how they planned their time, and whether they had developed the habit of meeting deadlines.

The good news is that habits can change. Your past behavior predicts your future behavior—but it doesn't determine it. Every assignment is a chance to build a new pattern.

If you've missed deadlines in the past, don't think of yourself as a 'late student.' Think of yourself as a student who is practicing a new habit. One on-time assignment becomes two. Two becomes five. Five becomes a track record.

#### What The Graph Doesn't Say

"The graph says I'm doomed because I've been late before."

#### It says:

"Your history predicts your future, but your actions today are how you change that history."

### Summary: Engineered Features

| Feature | Formula | What It Captures |
|---------|----------|------------------|
| `days_taken` | `date_submitted − date_assigned` | How long the student actually took, regardless of deadline |
| `time_per_question` | `time_spent ÷ num_questions` | Depth of engagement per item |
| `effort_ratio` | `time_spent ÷ prior_avg_time` | Whether the student worked more or less than usual |
| `time_pressure` | `difficulty ÷ days_until_due` | How challenging the assignment was relative to available time |
| `deadline_usage` | `days_taken ÷ days_until_due` | How much of the deadline window the student actually used |

> ⚠️ **Important note:** Some of these features — particularly `days_taken` and `deadline_usage` — are **post-hoc**: they can only be calculated *after* the assignment is submitted. They cannot be used to predict whether a student *will* be late before the deadline. We will need to be careful about this distinction when we build our model in Module 3.

---

## Part 4: Which Features Should We Actually Use?

Having identified which variables correlate with on-time completion, we now face a harder question: **just because a variable predicts something, does that mean we should use it?**

This is one of the most important questions in applied AI ethics, and it does not have a simple technical answer. It requires moral judgment.

### A Framework for Feature Evaluation

For each potential feature, we can ask three questions:

1. **Is it predictive?** Does it actually correlate with the outcome we're trying to predict?
2. **Is it appropriate?** Is it legitimate to use this information about a student for this purpose?
3. **Is it fair?** Could using this feature cause disproportionate harm to certain students?

Let's evaluate each feature in the dataset.

### The Two Most Ethically Complex Features

#### `prior_completion_rate`

This is the strongest predictor in the dataset (see the correlation chart above). Students who have been on time in the past are much more likely to be on time in the future. From a purely technical perspective, this is an excellent feature.

But consider what this feature actually encodes. A student's prior completion rate reflects not only their habits and effort but also their circumstances: access to a quiet place to work, family stability, health, whether they have after-school jobs or care responsibilities, prior teachers' grading policies, and many other factors entirely outside the student's control.

Using this feature as a predictor effectively **encodes the past into the future**. A student who has struggled is more likely to be flagged as at-risk going forward, regardless of whether anything has changed. This is the feedback loop problem described earlier.

#### `prior_avg_grade`

Grades are influenced by many factors beyond academic ability: access to tutoring, stable home environments, family educational background, quality of prior schooling, and language barriers, among others. A model that uses prior grades as a feature is, to some extent, a model that uses socioeconomic advantage as a feature — even if that was not the intent.

This is what *Magnifica Humanitas* (no. 103) means when it warns that when AI systems present themselves as neutral and objective, they can reflect and reinforce the stereotypes or ideological bias of their designers and developers. The bias is not always intentional. It is embedded in the data itself.

---

> 📝 **Reflection Exercise:** In the cells below, record your position on each of the two most ethically complex features.

### Should `prior_completion_rate` be used as a feature in our model?

Consider: it is the strongest predictor, but it may entrench disadvantage. Write your position and reasoning here.

*[Your answer here — try to argue both sides before reaching a conclusion]*

### Should `prior_avg_grade` be used as a feature in our model?

Consider: it correlates with on-time completion, but it may encode privilege. Write your position and reasoning here.

*[Your answer here]*

---

## Part 5: Producing Your Feature Analysis Report

For your project, you will produce a short written summary of your feature analysis. The code below generates a data-driven summary you can use as the foundation.

In [ ]:
# Generate a feature analysis summary
print("=" * 65)
print("FEATURE ANALYSIS REPORT")
print("Project: Predicting On-Time Homework Submission")
print("=" * 65)
print()

print("DATASET SUMMARY")
print(f"  Records:          {len(df)}")
print(f"  Students:         {df['student_id'].nunique()}")
print(f"  Assignments:      {df['assignment_id'].nunique()}")
print(f"  On-time rate:     {df['completed_on_time'].mean()*100:.1f}%")
print(f"  Late rate:        {(1 - df['completed_on_time'].mean())*100:.1f}%")
print()

print("TOP PREDICTIVE FEATURES (by correlation magnitude)")
top5 = correlations.abs().sort_values(ascending=False).head(5)
for feat, corr_abs in top5.items():
    actual = correlations[feat]
    direction = 'positive' if actual > 0 else 'negative'
    print(f"  {feat:<30} r = {actual:+.3f} ({direction})")
print()

print("FEATURES AVAILABLE BEFORE DEADLINE (usable for real prediction)")
pre_deadline = [
    'num_questions', 'difficulty', 'days_until_due',
    'prior_completion_rate', 'prior_avg_grade',
    'prior_avg_homework_time', 'time_pressure'
]
for f in pre_deadline:
    if f in correlations.index:
        print(f"  {f:<30} r = {correlations[f]:+.3f}")
print()

print("FEATURES FLAGGED FOR ETHICAL REVIEW")
print("  prior_completion_rate   — risk of feedback loop; encodes past circumstances")
print("  prior_avg_grade         — may encode socioeconomic advantage")
print()
print("=" * 65)

---

## Part 6: Deciding Your Feature Set

Based on your analysis in this module, you now need to make a decision that will carry forward into Module 3: **which features will you include in your model?**

There is no single correct answer. The goal is to make a reasoned, documented choice — balancing predictive power against ethical concerns — and to be able to defend it.

Use the cell below to record your decisions.

In [ ]:
# --- YOUR FEATURE DECISIONS ---
# Edit this list to reflect your choices for Module 3
# Only include features available BEFORE the deadline
# Add a comment explaining each decision

my_features = [
    'num_questions',           # Assignment property — no ethical concern
    'difficulty',              # Assignment property — no ethical concern
    'days_until_due',          # Assignment property — no ethical concern
    'prior_avg_homework_time', # Historical average behavior — low concern
    'time_pressure',           # Engineered: difficulty / days_until_due

    # DECIDE: include or exclude these?
    # 'prior_completion_rate',  # Strong predictor but feedback loop risk
    # 'prior_avg_grade',        # Predictive but may encode privilege
]

print("My chosen features for Module 3:")
for f in my_features:
    corr_val = correlations.get(f, 'N/A')
    print(f"  {f:<30} correlation = {corr_val}")
print()
print(f"Total features selected: {len(my_features)}")
print()
print("Features deliberately excluded:")
excluded = ['prior_completion_rate', 'prior_avg_grade',
            'time_spent_minutes', 'num_work_sessions',  # post-hoc
            'days_taken', 'time_per_question',          # post-hoc
            'effort_ratio', 'deadline_usage']           # post-hoc
for f in excluded:
    if f in correlations.index:
        print(f"  {f}")

### Justify Your Feature Choices

In the space below, write 3–5 sentences explaining your feature selection decisions. Address:
- Which features did you include and why?
- Which did you exclude, and what was your reasoning?
- Did you include or exclude `prior_completion_rate` and `prior_avg_grade`? Why?

*[Write your justification here]*

---

## Summary and What's Coming Next

### What You Accomplished in This Module

- Analyzed how on-time completion varies across student groups, assignment types, prior grades, and completion history
- Created **five engineered features** that capture patterns not visible in the raw data
- Identified that `prior_completion_rate` and `prior_avg_grade`, despite being strong predictors, raise significant ethical concerns around feedback loops and encoded privilege
- Made and documented a deliberate feature selection decision for your model

### Coming in Module 3 (Classes 5–6)

In Module 3 you will build your first actual predictive model:
- Prepare the data for modeling (encoding, scaling, train/test split)
- Train a simple **logistic regression** model — a transparent, interpretable algorithm well suited to this problem
- Evaluate its accuracy on held-out data
- Begin to understand *why* the model makes the predictions it does

The feature choices you made in this module will carry forward. If you want to change them later, that's fine — but documenting your reasoning now is part of what it means to build AI responsibly.

---

> **A thought to carry forward:** *Magnifica Humanitas* (no. 111) says that every design choice reflects a vision of humanity. The feature set you just chose is a design choice. The features you included say something about what you think is a fair basis for predicting a student's behavior. The features you excluded say something about what you think a student should not be judged by. That is a moral statement — whether or not it was framed that way.